# HEAD Adapted for YOLO26: YOLO26s → YOLO26n

Poltrada V3 · seed 42 · Kaggle GPU.

Tambahkan dataset, aktifkan GPU/Internet, dan secret `WANDB_API_KEY`. Run All dengan default menjalankan smoke test **1 epoch** pada 640/batch 16, termasuk validasi dan checkpoint. Ubah `RUN_FULL_TRAINING=True` untuk training setelah smoke test lulus.

Assistant adalah salinan Detect teacher yang trainable dan menerima neck student melalui adapter. AKD = detection loss assistant + mean MSE fitur intermediate assistant terhadap teacher. CKD = mean MSE fitur klasifikasi student terhadap teacher, melalui projector. Loss total = detection student + AKD + CKD. Ini adaptasi satu keluarga YOLO26, bukan reproduksi heterogeneous HEAD asli. [Paper HEAD](https://www.ecva.net/papers/eccv_2022/papers_ECCV/papers/136690310.pdf) / [kode penulis](https://github.com/LutingWang/HEAD).

## 1. Konfigurasi
Ubah parameter penelitian di cell ini.

In [ ]:
EXPERIMENT_ID = "13_head_yolo26n_teacher_yolo26s"
METHOD = "head"
MODEL_WEIGHTS = "yolo26n.pt"
DATA = "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
AMP = True
EVAL_SPLIT = "val"
PROJECT = "/kaggle/working/results"
RUN_FULL_TRAINING = False
METHOD_KWARGS = {'akd_weight': 1.0, 'ckd_weight': 1.0, 'assistant_det_weight': 1.0}
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_GROUP = "T26s_S26n"
TEACHER_ARTIFACT = "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
TEACHER_SHA256 = "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd"

## 2. Persiapan Kaggle
PyTorch GPU bawaan Kaggle tetap digunakan.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"{package}=={version}"])

import copy
import hashlib
import json
import math
import os
import shutil
import tempfile
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch import nn
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import ModelEMA, unwrap_model
import wandb

if metadata.version("ultralytics") != "8.4.155":
    raise RuntimeError("Restart kernel setelah instalasi Ultralytics.")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan GPU di Kaggle: Settings > Accelerator.")
settings.update({"wandb": False})  # Logging diatur oleh callback notebook.
if not os.environ.get("WANDB_API_KEY"):
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"])
if wandb.run is not None:
    raise RuntimeError("Selesaikan run W&B yang aktif sebelum menjalankan notebook ini.")

SESSION_DIR = Path(PROJECT) / f"{EXPERIMENT_ID}_{datetime.now():%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:6]}"
SESSION_DIR.mkdir(parents=True, exist_ok=False)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Output:", SESSION_DIR)

## 3. Dataset dan teacher
Split dan anotasi tetap. Teacher dikunci ke `best.pt` artifact v62; metadata epoch-nya 63, bukan 62.

In [ ]:
def class_names(names):
    return [str(names[k]) for k in sorted(names, key=lambda k: int(k))] if isinstance(names, dict) else list(names)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

data_info = check_det_dataset(DATA, autodownload=False)
if EVAL_SPLIT != "val" or not data_info.get("train") or not data_info.get("val"):
    raise ValueError("Gunakan split train dan val; test tidak dipakai untuk pemilihan checkpoint.")
DATASET_NAMES = class_names(data_info["names"])
DATASET_INFO = {"data": DATA, "names": DATASET_NAMES, "yaml_sha256": file_sha256(DATA)}
print("Kelas:", DATASET_NAMES)

In [ ]:
artifact = wandb.Api().artifact(TEACHER_ARTIFACT, type="model")
producer = artifact.logged_by()
if artifact.version != "v62" or producer is None or list(producer.path) != [WANDB_ENTITY, WANDB_PROJECT, "q5kp9k9d"]:
    raise ValueError("Artifact bukan teacher baseline YOLO26s yang ditetapkan.")
TEACHER_PATH = Path(artifact.get_path("best.pt").download(root=str(SESSION_DIR / "teacher")))
if file_sha256(TEACHER_PATH) != TEACHER_SHA256:
    raise ValueError("SHA-256 teacher berbeda; tidak ada fallback ke pretrained teacher.")
teacher = YOLO(str(TEACHER_PATH)).model
reference = DetectionModel("yolo26s.yaml", nc=data_info["nc"], verbose=False)
if teacher.yaml.get("scale") != "s" or class_names(teacher.names) != DATASET_NAMES:
    raise ValueError("Arsitektur atau urutan kelas teacher berbeda dari dataset.")
if {k: tuple(v.shape) for k, v in teacher.state_dict().items()} != {k: tuple(v.shape) for k, v in reference.state_dict().items()}:
    raise ValueError("Schema checkpoint bukan YOLO26s yang sesuai.")
TEACHER_INFO = {"artifact": TEACHER_ARTIFACT, "sha256": TEACHER_SHA256,
                "names": DATASET_NAMES, "artifact_metadata": dict(artifact.metadata)}
(SESSION_DIR / "teacher_provenance.json").write_text(json.dumps(TEACHER_INFO, indent=2))
print("Teacher:", artifact.name, "| epoch metadata:", artifact.metadata.get("epoch"))
del teacher, reference

## 4. Loss dan model
Detection loss resmi YOLO26 tetap digunakan pada kedua cabang. KD ditambahkan sekali dengan faktor batch. Modul tambahan HEAD hanya untuk training.

In [ ]:
"""HEAD Adapted for YOLO26. Training-only assistant; no PKD/native-KD objective.

Primary paper: https://www.ecva.net/papers/eccv_2022/papers_ECCV/papers/136690310.pdf
Official HEAD revision: 345fc77ae3d6c918cf9cdcbc0f1422b1afeb200f.
The one-stage formulation (paper Eq. 7) is adapted to YOLO26s -> YOLO26n.
"""

import math
from contextlib import contextmanager
from copy import deepcopy
from itertools import chain

import torch
import torch.nn.functional as F
from torch import nn
from ultralytics.utils.loss import E2ELoss

HEAD_SOURCE_COMMIT = "345fc77ae3d6c918cf9cdcbc0f1422b1afeb200f"
HEAD_TODD_COMMIT = "8ae5dbab32474880976dcdf15d508bb4925be1e2"


def head_feature_mse(pred, target):
    """Official HEAD/todd default: raw element-mean MSE, norm=False.

    FP32 arithmetic protects the reduction under AMP. No channel standardization,
    cosine normalization, spatial masking, score weighting, or PKD is performed.
    """
    if pred.shape != target.shape:
        raise ValueError(f"HEAD feature shape mismatch: {tuple(pred.shape)} != {tuple(target.shape)}")
    return F.mse_loss(pred.float(), target.detach().float(), reduction="mean")


def head_mean_feature_loss(predictions, targets):
    """Paper Eq. 5: average L feature pairs; each pair averages all elements."""
    if len(predictions) != len(targets) or not predictions:
        raise ValueError("HEAD needs the same nonzero number of feature pairs")
    return torch.stack([head_feature_mse(p, t) for p, t in zip(predictions, targets)]).mean()


def head_gradient_norm(parameters):
    grads = [p.grad.detach().float() for p in parameters if p.grad is not None]
    if not grads:
        return 0.0
    return float(torch.stack([g.square().sum() for g in grads]).sum().sqrt())


def _head_first_conv(module):
    return next(m for m in module.modules() if isinstance(m, nn.Conv2d))


def _head_out_channels(module):
    return [m.out_channels for m in module.modules() if isinstance(m, nn.Conv2d)][-1]


def _head_intermediates(head):
    # YOLO26 uses two internal stages and one final prediction conv per branch.
    return {
        f"{branch}_{level}_{stage}": pyramid[level][stage]
        for branch, pyramid in (("box", head.cv2), ("cls", head.cv3))
        for level in range(head.nl)
        for stage in range(len(pyramid[level]) - 1)
    }


@contextmanager
def _head_capture(head):
    """Ephemeral hooks; removed and tensor dictionaries cleared even on errors."""
    captured = {}
    handles = []
    try:
        def capture_inputs(module, inputs):
            captured["neck"] = tuple(inputs[0])

        handles.append(head.register_forward_pre_hook(capture_inputs))
        for key, stage in _head_intermediates(head).items():
            def capture_stage(module, inputs, output, name=key):
                captured[name] = output
            handles.append(stage.register_forward_hook(capture_stage))
        yield captured
    finally:
        for handle in handles:
            handle.remove()
        captured.clear()


class _HEADCriterionPair:
    """Keep official student/assistant E2E epoch schedules synchronized."""
    def __init__(self, student, assistant):
        self.student = student
        self.assistant = assistant

    def update(self):
        for criterion in (self.student, self.assistant):
            if hasattr(criterion, "update"):
                criterion.update()


class _HEADAssistantCriterionView(nn.Module):
    """Minimal model interface required by the official E2ELoss constructor."""
    def __init__(self, head, student):
        super().__init__()
        self.model = nn.ModuleList([head])
        self.args = student.args
        self.class_weights = getattr(student, "class_weights", None)

In [ ]:
class HEADModel(nn.Module):
    """Teacher-initialized assistant Detect + AKD + classification-only CKD.

    AKD uses both internal box/cls stages at all three levels of the O2M head.
    CKD maps the student's last internal O2M cls features to the teacher's. O2O
    remains fully supervised by the unmodified official detection criterion;
    its detached neck input cannot transfer assistant KD into the backbone.
    """
    def __init__(self, student_model, teacher_model, akd_weight=1.0,
                 ckd_weight=1.0, assistant_det_weight=1.0):
        super().__init__()
        for name, weight in (("akd_weight", akd_weight), ("ckd_weight", ckd_weight),
                             ("assistant_det_weight", assistant_det_weight)):
            if not math.isfinite(weight) or weight < 0:
                raise ValueError(f"{name} must be finite and nonnegative")
            setattr(self, name, float(weight))
        self.student_model = student_model
        self.teacher_model = teacher_model.to(next(student_model.parameters()).device)
        self._freeze_teacher()
        s_head, t_head = student_model.model[-1], teacher_model.model[-1]
        for head in (s_head, t_head):
            if head.nl != 3 or head.reg_max != 1:
                raise ValueError("HEAD adaptation requires unfused three-level YOLO26 reg_max=1 E2E Detect")
            if head.cv2 is None or head.cv3 is None or getattr(head, "one2one_cv2", None) is None:
                raise ValueError("Both official O2M and O2O branches are required")
            if any(len(branch) != 3 for branch in chain(head.cv2, head.cv3)):
                raise ValueError("Expected two intermediate stages before each YOLO26 prediction convolution")
            if tuple(float(v) for v in head.stride) != (8., 16., 32.):
                raise ValueError("HEAD adaptation requires P3/P4/P5 stride 8/16/32")
        if s_head.nc != t_head.nc:
            raise ValueError("Teacher and student class counts differ")
        self.assistant_head = deepcopy(t_head)
        self.assistant_head.requires_grad_(True)
        self.assistant_head.train(student_model.training)
        s_channels = [_head_first_conv(branch).in_channels for branch in s_head.cv2]
        t_channels = [_head_first_conv(branch).in_channels for branch in t_head.cv2]
        self.adapters = nn.ModuleList([
            nn.Conv2d(sc, tc, 1, bias=True) if sc != tc else nn.Identity()
            for sc, tc in zip(s_channels, t_channels)
        ])
        t_stages = _head_intermediates(t_head)
        self.akd_projectors = nn.ModuleDict({
            key: nn.Conv2d(_head_out_channels(stage), _head_out_channels(stage), 1, bias=True)
            for key, stage in t_stages.items()
        })
        self.ckd_projectors = nn.ModuleList([
            nn.Conv2d(_head_out_channels(s_head.cv3[i][-2]),
                      _head_out_channels(t_head.cv3[i][-2]), 1, bias=True)
            for i in range(s_head.nl)
        ])
        device = next(student_model.parameters()).device
        dtype = next(student_model.parameters()).dtype
        for module in (self.adapters, self.akd_projectors, self.ckd_projectors):
            module.to(device=device, dtype=dtype)
        self.assistant_criterion = None
        self._criterion_pair = None
        self.last_metrics = {}
        self.graph_audit = {
            "student_neck_indices": list(s_head.f), "teacher_neck_indices": list(t_head.f),
            "student_neck_channels": s_channels, "teacher_neck_channels": t_channels,
            "strides": [8, 16, 32], "reg_max": 1, "akd_feature_pairs": len(t_stages),
            "ckd_feature_pairs": 3, "kd_branch": "one2many",
            "student_cls_channels": [_head_out_channels(v[-2]) for v in s_head.cv3],
            "teacher_cls_channels": [_head_out_channels(v[-2]) for v in t_head.cv3],
        }

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            modules = self.__dict__.get("_modules", {})
            if "student_model" in modules:
                return getattr(modules["student_model"], name)
            raise

    @property
    def args(self):
        return self.student_model.args

    @args.setter
    def args(self, value):
        self.student_model.args = value

    @property
    def names(self):
        return self.student_model.names

    @names.setter
    def names(self, value):
        self.student_model.names = value

    @property
    def nc(self):
        return self.student_model.nc

    @nc.setter
    def nc(self, value):
        self.student_model.nc = value

    @property
    def criterion(self):
        if getattr(self.student_model, "criterion", None) is None:
            self.student_model.criterion = self.student_model.init_criterion()
        if self.assistant_criterion is None:
            self.assistant_criterion = E2ELoss(_HEADAssistantCriterionView(self.assistant_head, self.student_model))
        if self._criterion_pair is None:
            self._criterion_pair = _HEADCriterionPair(self.student_model.criterion, self.assistant_criterion)
        return self._criterion_pair

    @criterion.setter
    def criterion(self, value):
        self.student_model.criterion = value
        self.assistant_criterion = None
        self._criterion_pair = None

    def init_criterion(self):
        return self.criterion

    def _freeze_teacher(self):
        if self.teacher_model is not None:
            self.teacher_model.eval()
            self.teacher_model.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)
        self._freeze_teacher()
        return self

    def auxiliary_parameters(self):
        return chain(self.assistant_head.parameters(), self.adapters.parameters(),
                     self.akd_projectors.parameters(), self.ckd_projectors.parameters())

    def forward(self, x, *args, **kwargs):
        if isinstance(x, dict):
            return self.loss(x, *args, **kwargs)
        return self.student_model(x, *args, **kwargs)

    def loss(self, batch, preds=None):
        # Validator runs only the plain student. No teacher/assistant compute.
        if not self.training:
            regular, items = self.student_model.loss(batch, preds)
            zero = regular.new_zeros(1)
            return torch.cat((regular, zero)), {**items, "kd_loss": zero[0].detach()}
        if preds is not None:
            raise ValueError("HEAD training needs preds=None to capture intermediate features in a single forward")
        self._freeze_teacher()
        _ = self.criterion  # lazily create both official criteria on the final device
        image = batch["img"]
        batch_size = image.shape[0]
        s_head = self.student_model.model[-1]
        t_head = self.teacher_model.model[-1]
        with _head_capture(s_head) as student, _head_capture(t_head) as teacher, \
                _head_capture(self.assistant_head) as assistant:
            with torch.no_grad():
                self.teacher_model(image)
            predictions = self.student_model(image)
            regular, items = self.student_model.loss(batch, predictions)
            neck = student["neck"]
            if any(s.shape[-2:] != t.shape[-2:] for s, t in zip(neck, teacher["neck"])):
                raise ValueError("Teacher/student spatial grids differ; no hidden interpolation is allowed")
            assistant_predictions = self.assistant_head([adapter(feat) for adapter, feat in zip(self.adapters, neck)])
            assistant_vector, _ = self.assistant_criterion(assistant_predictions, batch)
            # Official criterion already multiplies by B: undo once before forming KD.
            assistant_det = assistant_vector.sum() / batch_size
            # Projectors and reductions are FP32 outside AMP. Autograd crosses casts.
            with torch.autocast(device_type=image.device.type, enabled=False):
                projected = [self.akd_projectors[key](assistant[key].float()) for key in self.akd_projectors]
                akd = head_mean_feature_loss(projected, [teacher[key] for key in self.akd_projectors])
                ckd = head_mean_feature_loss(
                    [projector(student[f"cls_{i}_1"].float()) for i, projector in enumerate(self.ckd_projectors)],
                    [teacher[f"cls_{i}_1"] for i in range(3)],
                )
            kd = self.assistant_det_weight * assistant_det + self.akd_weight * akd + self.ckd_weight * ckd
            det = regular.sum() / batch_size
            self.last_metrics = {
                "det_loss": float(det.detach()), "kd_loss": float(kd.detach()),
                "total_loss": float((det + kd).detach()), "akd_feature_loss": float(akd.detach()),
                "assistant_det_loss": float(assistant_det.detach()), "ckd_loss": float(ckd.detach()),
                "kd_det_ratio": float(kd.detach() / det.detach().clamp_min(1e-12)),
            }
            return torch.cat((regular, (kd * batch_size).reshape(1))), {**items, "kd_loss": kd.detach()}

    def fuse(self, verbose=True, imgsz=640):
        return self.student_model.fuse(verbose=verbose, imgsz=imgsz)


KDModel = HEADModel

## 5. Trainer dan checkpoint
EMA, validasi, dan `best.pt` memakai student biasa. State training tambahan disimpan terpisah; resume belum diverifikasi.

In [ ]:
def teacher_digest(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def gradient_norm(parameters):
    values = [p.grad.detach().float().norm() for p in parameters if p.grad is not None]
    result = float(torch.stack(values).norm()) if values else 0.0
    if not math.isfinite(result) or result <= 0:
        raise RuntimeError("Gradien harus finite dan nonzero.")
    return result

class StudentEMA(ModelEMA):
    def __init__(self, model):
        super().__init__(unwrap_model(model).student_model)
    def update(self, model):
        super().update(unwrap_model(model).student_model)
    def update_attr(self, model, include=(), exclude=()):
        super().update_attr(unwrap_model(model).student_model, include, exclude)

class ResearchTrainer(DetectionTrainer):
    def set_model_attributes(self):
        super().set_model_attributes()
        if self.args.resume or self.args.compile or self.world_size > 1:
            raise ValueError("Gunakan satu GPU, compile=False, resume=False; RESUME_NOT_VERIFIED.")
        teacher = YOLO(str(TEACHER_PATH)).model.to(self.device).eval().requires_grad_(False)
        self.model = KDModel(self.model, teacher, **METHOD_KWARGS).to(self.device)

    def build_optimizer(self, model, *args, **kwargs):
        model.teacher_model.eval().requires_grad_(False)
        optimizer = super().build_optimizer(model, *args, **kwargs)
        for group in optimizer.param_groups:
            group["params"] = [p for p in group["params"] if p.requires_grad]
        actual = [id(p) for group in optimizer.param_groups for p in group["params"]]
        expected = {id(p) for p in model.parameters() if p.requires_grad}
        if set(actual) != expected or len(actual) != len(set(actual)):
            raise RuntimeError("Parameter optimizer tidak sesuai.")
        return optimizer

    def _setup_train(self):
        super()._setup_train()
        self.ema = None
        self.ema = StudentEMA(self.model)

    def _build_train_pipeline(self):
        if self.batch_size != BATCH:
            raise RuntimeError("Batch berubah setelah OOM; hentikan dan tinjau konfigurasi.")
        super()._build_train_pipeline()

    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)
        self.actual_shape = tuple(batch["img"].shape)
        return batch

    def _handle_nan_recovery(self, epoch):
        if not torch.isfinite(self.loss).all() or not math.isfinite(float(self.fitness)):
            raise RuntimeError("Loss/fitness non-finite; training dihentikan.")
        return False

    def save_model(self):
        # Hanya DetectionModel biasa yang masuk best.pt/last.pt.
        student = copy.deepcopy(self.ema.ema).half().to(memory_format=torch.contiguous_format)
        student.criterion = None
        if not all(torch.isfinite(v).all() for v in student.state_dict().values() if v.is_floating_point()):
            raise RuntimeError("Checkpoint student mengandung NaN/Inf.")
        checkpoint = {"model": student, "ema": None, "optimizer": None, "epoch": int(self.epoch),
                      "best_fitness": float(self.best_fitness), "updates": self.ema.updates,
                      "train_args": vars(self.args), "train_metrics": {**self.metrics, "fitness": self.fitness},
                      "train_results": self.read_results_csv(), "version": metadata.version("ultralytics"),
                      "method": METHOD, "teacher_provenance": TEACHER_INFO, "resume_status": "INFERENCE_ONLY"}
        self.wdir.mkdir(parents=True, exist_ok=True)
        torch.save(checkpoint, self.last)
        if self.fitness == self.best_fitness:
            torch.save(checkpoint, self.best)
        return True

## 6. Logging W&B
Loss per komponen, mAP, precision/recall, LR, gradien, VRAM, dan durasi dicatat. Snapshot setiap epoch baru dibersihkan setelah upload selesai.

In [ ]:
def cpu_state(state):
    return {k: v.detach().cpu().clone() for k, v in state.items()}

def make_callbacks(run, diagnostic):
    context = {"steps": 0, "optimizer_steps": 0, "best_epoch": None, "sums": {}, "batches": 0}

    def start(trainer):
        context["teacher_before"] = teacher_digest(trainer.model.teacher_model)
        context["started"] = time.perf_counter()
        if trainer.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(trainer.device)
        def before_step(optimizer, *_):
            # Fused AdamW memanggil hook juga saat GradScaler melewati update.
            found_inf = getattr(optimizer, "found_inf", None)
            if found_inf is not None and float(found_inf) != 0:
                context["amp_skipped_steps"] = context.get("amp_skipped_steps", 0) + 1
                return
            context["student_grad"] = gradient_norm(trainer.model.student_model.parameters())
            if hasattr(trainer.model, "auxiliary_parameters"):
                context["aux_grad"] = gradient_norm(trainer.model.auxiliary_parameters())
            context["optimizer_steps"] += 1
        context["gradient_hook"] = trainer.optimizer.register_step_pre_hook(before_step)

    def batch_start(trainer):
        trainer._oom_retries = 3  # Tidak mengizinkan native OOM batch-halving.
        if trainer.batch_size != BATCH or int(trainer.args.imgsz) != IMGSZ or bool(trainer.amp) != AMP:
            raise RuntimeError("Batch, resolusi, atau AMP efektif berbeda dari konfigurasi.")

    def batch_end(trainer):
        context["steps"] += 1
        context["batches"] += 1
        for key, value in trainer.model.last_metrics.items():
            if not math.isfinite(float(value)):
                raise RuntimeError(f"Metrik non-finite: {key}")
            context["sums"][key] = context["sums"].get(key, 0) + float(value)
        teacher = trainer.model.teacher_model
        if teacher.training or any(p.grad is not None for p in teacher.parameters()):
            raise RuntimeError("Teacher keluar dari eval atau menerima gradien.")

    def save(trainer):
        epoch = int(trainer.epoch) + 1
        if trainer.fitness == trainer.best_fitness:
            context["best_epoch"] = epoch
        state = {"epoch": epoch, "student": cpu_state(trainer.model.student_model.state_dict()),
                 "auxiliary": cpu_state({k: v for k, v in trainer.model.state_dict().items()
                                         if not k.startswith(("student_model.", "teacher_model."))}),
                 "optimizer": trainer.optimizer.state_dict(), "scaler": trainer.scaler.state_dict(),
                 "resume_status": "RESUME_NOT_VERIFIED"}
        torch.save(state, trainer.wdir / "training_state_last.pt")
        if trainer.fitness == trainer.best_fitness:
            shutil.copy2(trainer.wdir / "training_state_last.pt", trainer.wdir / "training_state_best.pt")
        # Snapshot tetap utuh sampai upload selesai, lalu file sementara dibersihkan.
        with tempfile.TemporaryDirectory(prefix=f"epoch-{epoch:03d}-", dir=str(trainer.save_dir)) as temporary:
            snapshot = Path(temporary)
            item = wandb.Artifact(f"{EXPERIMENT_ID}-{run.id}-checkpoints", type="model",
                    metadata={"epoch": epoch, "best_epoch": context["best_epoch"], "method": METHOD,
                              "diagnostic": diagnostic, "teacher_sha256": TEACHER_SHA256,
                              "resume_status": "RESUME_NOT_VERIFIED"})
            for filename in ("last.pt", "best.pt", "training_state_last.pt", "training_state_best.pt"):
                shutil.copy2(trainer.wdir / filename, snapshot / filename)
                item.add_file(str(snapshot / filename), name=filename)
            run.log_artifact(item, aliases=["latest", f"epoch-{epoch:03d}"] +
                             (["best"] if trainer.fitness == trainer.best_fitness else [])).wait()
        values = {f"train/{k}": v / context["batches"] for k, v in context["sums"].items()}
        values.update(trainer.metrics)
        values.update(trainer.lr)
        values.update(epoch=epoch, best_epoch=context["best_epoch"],
                      elapsed_seconds=time.perf_counter() - context["started"],
                      amp_skipped_steps=context.get("amp_skipped_steps", 0),
                      student_gradient_norm=context.get("student_grad", 0),
                      auxiliary_gradient_norm=context.get("aux_grad", 0))
        if trainer.device.type == "cuda":
            values.update(peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
                          peak_reserved_gib=torch.cuda.max_memory_reserved() / 2**30)
        run.log(values, step=epoch)
        context["sums"], context["batches"] = {}, 0

    return {"on_train_start": start, "on_train_batch_start": batch_start,
            "on_train_batch_end": batch_end, "on_model_save": save}, context

In [ ]:
def experiment_config():
    return {"method": METHOD, "model": MODEL_WEIGHTS, "data": DATA, "epochs": EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "optimizer": OPTIMIZER, "lr0": LR,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC, "amp": AMP, "workers": WORKERS,
            "method_kwargs": METHOD_KWARGS, "teacher_sha256": TEACHER_SHA256}

def run_training(diagnostic=False):
    if not 1 <= EPOCHS <= 100:
        raise ValueError("Gunakan 1 sampai 100 epoch.")
    config = experiment_config()
    name = ("diagnostic" if diagnostic else "training") + "_" + uuid.uuid4().hex[:6]
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                     name=f"{EXPERIMENT_ID}_seed{SEED}_{name}", id=uuid.uuid4().hex[:8], resume="never",
                     tags=[METHOD, "teacher-s", "student-n", f"seed{SEED}", "poltrada-v3", name.split("_")[0]],
                     config={**config, "dataset": DATASET_INFO, "teacher": TEACHER_INFO}, dir=str(SESSION_DIR))
    callbacks, context = make_callbacks(run, diagnostic)
    trainer = None
    try:
        trainer = ResearchTrainer(overrides={"model": MODEL_WEIGHTS, "data": DATA, "epochs": 1 if diagnostic else EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": DEVICE, "workers": WORKERS,
            "optimizer": OPTIMIZER, "lr0": LR, "weight_decay": WEIGHT_DECAY,
            "warmup_epochs": WARMUP_EPOCHS, "close_mosaic": CLOSE_MOSAIC, "amp": AMP,
            "patience": 100, "split": EVAL_SPLIT, "project": str(SESSION_DIR), "name": name,
            "plots": not diagnostic, "save": True, "exist_ok": False, "resume": False,
            "compile": False, "distill_model": None})
        for event, callback in callbacks.items():
            trainer.add_callback(event, callback)
        trainer.train()
        if context["optimizer_steps"] < (2 if diagnostic else 1):
            raise RuntimeError("Belum ada cukup optimizer step nyata yang berhasil.")
        if teacher_digest(trainer.model.teacher_model) != context["teacher_before"]:
            raise RuntimeError("Parameter atau BatchNorm teacher berubah.")
        if not all(math.isfinite(float(v)) for v in trainer.metrics.values()):
            raise RuntimeError("Metrik validasi non-finite.")
        student = YOLO(str(trainer.best)).model.eval()
        if student.yaml.get("scale") != "n" or hasattr(student, "teacher_model"):
            raise RuntimeError("Checkpoint inferensi bukan student-only.")
        result = {"status": "PASS", "config": config, "epochs_this_run": 1 if diagnostic else EPOCHS,
                  "training_batches": context["steps"], "best_pt": str(trainer.best),
                  "best_epoch": context["best_epoch"], "metrics": trainer.metrics,
                  "optimizer_steps": context["optimizer_steps"],
                  "seconds": time.perf_counter() - context["started"], "resume": "RESUME_NOT_VERIFIED"}
        (Path(trainer.save_dir) / "summary.json").write_text(json.dumps(result, indent=2))
        run.summary.update(result)
        run.finish(exit_code=0)
        return result
    except BaseException:
        run.finish(exit_code=1)
        raise
    finally:
        if context.get("gradient_hook"):
            context["gradient_hook"].remove()

## 7. Smoke test Kaggle — 1 epoch
Satu epoch pada seluruh split train, lalu validasi dan load ulang student. Jika OOM/AMP/loss gagal, training berhenti tanpa menurunkan batch atau resolusi otomatis.

In [ ]:
def numerical_verification():
    """Hand-computed raw-MSE oracle, averaging, target gradient isolation."""
    pred = torch.tensor([[[[1., 3.]]]], requires_grad=True)
    target = torch.tensor([[[[3., 1.]]]], requires_grad=True)
    mse = head_feature_mse(pred, target)
    torch.testing.assert_close(mse, torch.tensor(4.))
    mse.backward()
    torch.testing.assert_close(pred.grad, torch.tensor([[[[-2., 2.]]]]))
    assert target.grad is None
    mixed = head_mean_feature_loss([pred, 2 * pred], [target, target])
    torch.testing.assert_close(mixed, torch.tensor(8.5))
    # Offsets remain meaningful: standardizing features would incorrectly erase them.
    shifted = head_feature_mse(torch.ones(1, 2, 2, 2), torch.zeros(1, 2, 2, 2))
    torch.testing.assert_close(shifted, torch.tensor(1.))
    return {"raw_mse_oracle": "PASS CPU", "stage_average": "PASS CPU",
            "target_gradient_isolation": "PASS CPU", "no_feature_normalization": "PASS CPU"}

print(numerical_verification())

In [ ]:
DIAGNOSTIC_RESULT = run_training(diagnostic=True)
print(json.dumps(DIAGNOSTIC_RESULT, indent=2))

## 8. Training dan hasil
Satu metode, satu seed, maksimal 100 epoch. Untuk uji satu epoch saja, set `EPOCHS=1`.

In [ ]:
TRAINING_RESULT = None
if RUN_FULL_TRAINING:
    if DIAGNOSTIC_RESULT["status"] != "PASS" or DIAGNOSTIC_RESULT["config"] != experiment_config():
        raise RuntimeError("Jalankan ulang smoke test setelah mengubah konfigurasi.")
    TRAINING_RESULT = run_training(diagnostic=False)
else:
    print("Smoke test selesai. Full training belum dijalankan.")

In [ ]:
result = TRAINING_RESULT or DIAGNOSTIC_RESULT
print("Checkpoint student:", result["best_pt"])
print("Best epoch:", result["best_epoch"])
print("Validation:", result["metrics"])
print("Durasi (detik):", round(result["seconds"], 2))